# S20 — Decoding, Prompting, and In-Context Learning

**Week 11 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s20_decoding_prompting_and_in_context_learning.ipynb)

Every cell below is a worked example from the [S20 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s20/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s20.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s20.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## From distribution to token: the decoding zoo


*Expected output starts with:* `true distribution:`


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# A fixed toy next-token distribution: pretend the model just read "The cat"
vocab = ["sat", "ran", "slept", "is", "jumped", "flew", "quantum", "lasagna"]
logits = torch.tensor([2.0, 1.5, 1.2, 0.8, 0.5, -0.5, -2.0, -3.0])
probs = F.softmax(logits, dim=-1)
print("true distribution:")
for tok, p in zip(vocab, probs):
    print(f"  {tok:>8}: {p.item():.4f}")

def sample(logits, n, temperature=1.0, top_k=None, top_p=None):
    logits = logits / temperature
    if top_k is not None:
        kth = torch.topk(logits, top_k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        # keep the smallest prefix whose cumulative probability reaches top_p
        cut = torch.searchsorted(cum, torch.tensor(top_p)).item() + 1
        keep = sorted_idx[:cut]
        mask = torch.full_like(logits, float("-inf"))
        mask[keep] = logits[keep]
        logits = mask
    p = F.softmax(logits, dim=-1)
    return torch.multinomial(p, n, replacement=True)

def freq_table(samples):
    counts = torch.bincount(samples, minlength=len(vocab)).float() / len(samples)
    return "  ".join(f"{vocab[i]}:{counts[i]:.3f}"
                     for i in range(len(vocab)) if counts[i] > 0)

N = 10000
print(f"\ngreedy: always '{vocab[int(torch.argmax(logits))]}'")
for T in [0.5, 1.0, 2.0]:
    s = sample(logits, N, temperature=T)
    print(f"T={T:>3}: {freq_table(s)}")
s = sample(logits, N, top_k=3)
print(f"top-k (k=3):  {freq_table(s)}")
s = sample(logits, N, top_p=0.9)
print(f"top-p (p=0.9): {freq_table(s)}")

## Getting top-p right


*Expected output starts with:* `peaked distribution [0.9700, 0.0150, 0.0080, 0.0040, 0.0030], p=0.9`


In [ ]:
import torch

torch.manual_seed(0)

def top_p_filter_buggy(probs, p):
    """Keep tokens while cumulative probability stays BELOW p. Looks right; isn't."""
    sorted_p, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    keep = sorted_idx[cum < p]
    out = torch.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum() if out.sum() > 0 else out

def top_p_filter(probs, p):
    """Smallest prefix of the sorted tokens whose cumulative probability >= p."""
    sorted_p, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    # shift: a token is removed only if the mass BEFORE it already reached p,
    # so the top token always survives
    remove = cum - sorted_p >= p
    keep = sorted_idx[~remove]
    out = torch.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

def show(q):
    return "[" + ", ".join(f"{v:.4f}" for v in q.tolist()) + "]"

vocab = ["yes", "no", "maybe", "perhaps", "possibly"]
p = 0.9

# Edge case: one token already carries more mass than p
probs = torch.tensor([0.97, 0.015, 0.008, 0.004, 0.003])
print(f"peaked distribution {show(probs)}, p={p}")
print(f"  buggy   -> {show(top_p_filter_buggy(probs, p))}")
print(f"  correct -> {show(top_p_filter(probs, p))}")

# Boundary case: the token that crosses p must stay in the nucleus
probs = torch.tensor([0.40, 0.25, 0.20, 0.10, 0.05])
print(f"\nflat distribution {show(probs)}, p={p}")
for name, fn in [("buggy", top_p_filter_buggy), ("correct", top_p_filter)]:
    q = fn(probs, p)
    kept = [vocab[i] for i in range(len(vocab)) if q[i] > 0]
    print(f"  {name:>7} keeps {kept} -> {show(q)}")

## Beam search and the length problem


*Expected output starts with:* `greedy (width 1) : 'the sat'  logp = -1.9841`


In [ ]:
# Beam search on a hand-specified bigram language model, small enough to
# check against brute-force enumeration. Shows (1) greedy missing the best
# sequence, (2) why raw log-probability favors short outputs, and (3) the
# GNMT length penalty fixing the ranking.
import math
from itertools import product

# next-token distributions, conditioned on the previous token only
model = {
    "<s>":    {"the": 0.55, "a": 0.45},
    "the":    {"man": 0.25, "boat": 0.25, "sat": 0.25, "old": 0.25},
    "a":      {"boat": 0.90, "man": 0.05, "sat": 0.05},
    "man":    {"<eos>": 0.90, "sat": 0.10},
    "old":    {"man": 0.90, "<eos>": 0.10},
    "sat":    {"<eos>": 1.00},
    "boat":   {"<eos>": 0.50, "sailed": 0.50},
    "sailed": {"away": 0.95, "<eos>": 0.05},
    "away":   {"<eos>": 0.95, "sailed": 0.05},
}
MAX_LEN = 6

def logp(seq):
    """Total log-probability of a complete sequence (ends with <eos>)."""
    total, prev = 0.0, "<s>"
    for tok in seq:
        total += math.log(model[prev][tok])
        prev = tok
    return total

def beam_search(width):
    beams = [(0.0, ["<s>"])]
    finished = []
    for _ in range(MAX_LEN):
        candidates = []
        for score, seq in beams:
            for tok, p in model[seq[-1]].items():
                cand = (score + math.log(p), seq + [tok])
                (finished if tok == "<eos>" else candidates).append(cand)
        beams = sorted(candidates, reverse=True)[:width]
        if not beams:
            break
    return sorted(finished, reverse=True)

def gnmt_penalty(seq, alpha=0.7):
    """Wu et al. (2016): divide log-prob by ((5 + |Y|) / 6)^alpha."""
    n_tokens = len(seq) - 1                      # exclude <s>... count words+eos
    return ((5 + n_tokens) / 6) ** alpha

def show(seq):
    return " ".join(seq[1:-1])

# greedy = beam search with width 1
greedy = beam_search(1)[0]
wide = beam_search(3)
print(f"greedy (width 1) : '{show(greedy[1])}'  logp = {greedy[0]:.4f}")
print(f"beam (width 3)   : '{show(wide[0][1])}'  logp = {wide[0][0]:.4f}")

# brute force: enumerate every sequence up to MAX_LEN to find the true argmax
best = max(
    ((logp(list(seq) + ["<eos>"]), ["<s>"] + list(seq) + ["<eos>"])
     for L in range(1, MAX_LEN)
     for seq in product([t for t in model if t != "<s>"], repeat=L)
     if all(b in model.get(a, {}) for a, b in zip(["<s>"] + list(seq), list(seq) + ["<eos>"]))),
    key=lambda x: x[0])
print(f"exhaustive best  : '{show(best[1])}'  logp = {best[0]:.4f}")

print(f"\ntop finished hypotheses, two rankings (alpha = 0.7):")
print(f"{'sequence':>24} {'logp':>9} {'logp/penalty':>13}")
for score, seq in wide[:4]:
    print(f"{show(seq):>24} {score:>9.4f} {score / gnmt_penalty(seq):>13.4f}")
reranked = sorted(wide[:4], key=lambda x: x[0] / gnmt_penalty(x[1]), reverse=True)
print(f"\nraw ranking picks      : '{show(wide[0][1])}'")
print(f"length-penalty ranking : '{show(reranked[0][1])}'")

## Speculative decoding: faster sampling, same distribution


*Expected output starts with:* `target p = ['0.55', '0.25', '0.10', '0.06', '0.04']`


In [ ]:
# Speculative sampling in miniature. A cheap draft model q proposes a token;
# the target model p accepts it with probability min(1, p(x)/q(x)); on
# rejection we sample from the residual max(p - q, 0), renormalized.
# Claim to verify: the output distribution is EXACTLY p, for any q.
import torch

torch.manual_seed(0)

p = torch.tensor([0.55, 0.25, 0.10, 0.06, 0.04])   # target model's next-token dist

def speculative_step(p, q, gen):
    x = torch.multinomial(q, 1, generator=gen).item()      # draft proposes
    if torch.rand(1, generator=gen).item() < min(1.0, (p[x] / q[x]).item()):
        return x, True                                     # target accepts
    residual = (p - q).clamp(min=0)                        # else resample from
    residual /= residual.sum()                             # normalized excess
    return torch.multinomial(residual, 1, generator=gen).item(), False

drafts = {
    "q = p (perfect draft)": p.clone(),
    "q close to p":          torch.tensor([0.45, 0.30, 0.12, 0.08, 0.05]),
    "q far from p":          torch.tensor([0.05, 0.05, 0.10, 0.30, 0.50]),
    "q uniform":             torch.full((5,), 0.2),
}

N = 200_000
gen = torch.Generator().manual_seed(1)
print(f"target p = {[f'{v:.2f}' for v in p.tolist()]}")
print(f"{'draft model':>22} {'accept rate':>12} {'1 - TV(p,q)':>12} {'max |freq - p|':>15}")
for name, q in drafts.items():
    counts = torch.zeros(5)
    accepted = 0
    for _ in range(N):
        x, ok = speculative_step(p, q, gen)
        counts[x] += 1
        accepted += ok
    freqs = counts / N
    theory = torch.minimum(p, q).sum().item()   # E[accept] = sum_x min(p, q)
    print(f"{name:>22} {accepted / N:>12.4f} {theory:>12.4f} "
          f"{(freqs - p).abs().max().item():>15.4f}")

# Why acceptance rate is the whole game: with a draft of gamma tokens per
# target-model call, the expected number of tokens produced per call is
# (1 - a^(gamma+1)) / (1 - a) for per-token acceptance rate a.
print(f"\nexpected tokens per target-model call (draft length gamma):")
print(f"{'a':>6} {'gamma=2':>8} {'gamma=4':>8} {'gamma=8':>8}")
for a in [0.5, 0.7, 0.9]:
    row = [(1 - a ** (g + 1)) / (1 - a) for g in (2, 4, 8)]
    print(f"{a:>6} {row[0]:>8.2f} {row[1]:>8.2f} {row[2]:>8.2f}")

## Try it yourself

1. In the beam-search script, sweep the width over 1, 2, 3, 5 and the length-penalty `alpha` over 0.0, 0.7, 1.5, printing the winning sequence for each combination. At what width does the search recover the exhaustive optimum, and at what `alpha` does the ranking start favoring the longest hypothesis regardless of probability?
2. Extend the decoding script to compute the empirical entropy of the samples at each temperature (`-sum p*log p` of the frequency table) and plot or print entropy versus `T` for `T` in `{0.25, 0.5, 1, 2, 4}`.
3. Implement *min-p* sampling (keep tokens whose probability is at least `min_p` times the top token's probability) and compare the kept sets against top-p on both distributions from the edge-case script.
4. Combine filters in both orders — top-k-then-temperature versus temperature-then-top-k — on the toy distribution with `k = 3`, `T = 2.0`. Do the empirical frequencies differ, and can you explain why?


---

Full discussion of everything above: [S20 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s20/).
